In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

## social network ads dataset

In [3]:
df = pd.read_csv("./Social_Network_Ads.csv")

In [4]:
df.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [5]:
from sklearn.preprocessing import LabelEncoder

In [6]:
le = LabelEncoder()

In [7]:
df['Gender'] = le.fit_transform(df['Gender'])

In [8]:
df.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,1,19,19000,0
1,15810944,1,35,20000,0
2,15668575,0,26,43000,0
3,15603246,0,27,57000,0
4,15804002,1,19,76000,0


In [9]:
X = df.drop(columns=['User ID','Purchased'])
y = df['Purchased']

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
from sklearn.preprocessing import StandardScaler

In [12]:
scaler = StandardScaler()

In [13]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
knn = KNeighborsClassifier(n_neighbors=8)

In [15]:
knn.fit(X_train,y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",8
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [16]:
y_pred = knn.predict(X_test)

In [17]:
accuracy_score(y_test,y_pred)

0.9375

In [18]:
for i in range(1,15):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train,y_train)
    print(i,accuracy_score(y_test,knn.predict(X_test)))

1 0.8875
2 0.8375
3 0.9125
4 0.9125
5 0.925
6 0.925
7 0.925
8 0.9375
9 0.9375
10 0.9375
11 0.9375
12 0.9375
13 0.9375
14 0.9375


In [19]:
from collections import Counter

In [20]:
class MeraKnn:
    def __init__(self,k=5):
        self.n_neighbors = k
        self.X_train = None
        self.y_train = None

    # calculate distance 
    def caculate_distance(slef,PointA,PointB):
        return np.linalg.norm(PointA - PointB)

    # count most common value
    def majority_count(self,neighbors):
        votes = []
        for i in neighbors:
            votes.append(self.y_train.iloc[i[0]])
        Votes = Counter(votes)
        return Votes.most_common()[0][0]
    
    def fit(self,X_train,y_train):
        self.X_train = X_train
        self.y_train = y_train
 
    def predict(self,X_test):
        y_pred = []
        for i in X_test: 
            #calculate distance
            distances = []
            for j in self.X_train:
                distances.append(self.caculate_distance(i,j))
                
            neighbors = sorted(
                list(
                    enumerate(distances)
                ),key=lambda x : x[1]
            )[0:self.n_neighbors]
            # sort value for second item 
            # enumerate is return index with value tuple 
            lable = self.majority_count(neighbors)
            y_pred.append(lable)
        return np.array(y_pred)             

In [21]:
meraKnn = MeraKnn(k=8)

In [22]:
meraKnn.fit(X_train,y_train)

In [23]:
y_pred1 = meraKnn.predict(X_test)

In [24]:
 accuracy_score(y_test,y_pred)

0.9375

In [25]:
class KNN:
    def __init__(self,k=5):
        self.n_neighbors = k
        self.X_train = None
        self.y_train = None

    def fit(self,X_train,y_train):
        self.X_train = X_train.values if hasattr(X_train, "values") else X_train
        self.y_train = y_train

    def calcualte_distance(self,pointA,pointB):
        return np.linalg.norm(pointA - pointB)

    def majority_count(self,neighbors):
        votes = []
        for idx,dist in neighbors:
            votes.append(self.y_train.iloc[idx])
        Votes = Counter(votes)
        return Votes.most_common(1)[0][0]
        
    def score(self, X_test, y_test):
        y_pred = self.predict(X_test)
        return np.mean(y_pred == y_test)

    def predict(self,X_test):
        y_prad = []
        for test_point in X_test:
            distance = []
            for i,train_point in enumerate(self.X_train):
                dist = self.calcualte_distance(test_point,train_point)
                distance.append((i,dist))

            neighbors = sorted(distance,key=lambda x : x[1])[:self.n_neighbors]
                
            label = self.majority_count(neighbors)
            y_prad.append(label)
        return np.array(y_prad)

In [26]:
knn = KNN(k=8)

In [27]:
knn.fit(X_train,y_train)

In [28]:
y_pred = knn.predict(X_test)

In [29]:
accuracy_score(y_test,y_pred)

0.9375

In [30]:
knn.score(X_test,y_test)

np.float64(0.9375)

In [52]:
class FastKNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = X.values if hasattr(X, "values") else X
        self.y_train = y.values if hasattr(y, "values") else y

    def predict(self, X_test):
        X_test = X_test.values if hasattr(X_test, "values") else X_test

        # 🔥 Vectorized distance (no loops)
        distances = np.sqrt(
            np.sum((X_test[:, np.newaxis] - self.X_train) ** 2, axis=2)
        )

        # get k nearest indices
        k_idx = np.argsort(distances, axis=1)[:, :self.k]

        y_pred = []

        for idx in k_idx:
            labels = self.y_train[idx]
            y_pred.append(Counter(labels).most_common(1)[0][0])

        return np.array(y_pred)

In [61]:
knn = FastKNN(8)

In [62]:
knn.fit(X_train,y_train)

In [63]:
y_pred = knn.predict(X_test)

In [64]:
accuracy_score(y_test,y_pred)

0.9375